In [1]:
import torch
print(torch.cuda.is_available())

True


In [2]:
import pandas as pd
from transformers import T5Tokenizer, Trainer, TrainingArguments, T5ForConditionalGeneration

In [3]:
train_data = pd.read_csv("samsum-train.csv")
val_data = pd.read_csv("samsum-validation.csv")

In [4]:
train_data.head()

,id,dialogue,summary
0,13818513,Amanda: I baked cookies. Do you want some?\r\...,Amanda baked cookies and will bring Jerry some...
1,13728867,Olivia: Who are you voting for in this electio...,Olivia and Olivier are voting for liberals in ...
2,13681000,"Tim: Hi, what's up?\r\nKim: Bad mood tbh, I wa...",Kim may try the pomodoro technique recommended...
3,13730747,"Edward: Rachel, I think I'm in ove with Bella....",Edward thinks he is in love with Bella. Rachel...
4,13728094,Sam: hey overheard rick say something\r\nSam:...,"Sam is confused, because he overheard Rick com..."


In [5]:
train_data["dialogue"][0:10]

0    Amanda: I baked  cookies. Do you want some?\r\...
1    Olivia: Who are you voting for in this electio...
2    Tim: Hi, what's up?\r\nKim: Bad mood tbh, I wa...
3    Edward: Rachel, I think I'm in ove with Bella....
4    Sam: hey  overheard rick say something\r\nSam:...
5    Neville: Hi there, does anyone remember what d...
6    John: Ave. Was there any homework for tomorrow...
7    Sarah: I found a song on youtube and I think y...
8    Noah: When and where are we meeting? :)\r\nMad...
9    Matt: Do you want to go for date?\r\nAgnes: Wo...
Name: dialogue, dtype: object

In [6]:
val_data.shape

(818, 3)

In [7]:
train_data.shape

(14732, 3)

In [8]:
# Random Sampling
train_data = train_data.sample(n=4000, random_state=42).reset_index(drop=True)
val_data = val_data.sample(n=500, random_state=42).reset_index(drop=True)

In [9]:
train_data.shape
val_data.shape

(500, 3)

# Data pre-processing

In [10]:
import re

def clean_data(text):
    text = re.sub(r"\r\n", " ",text)
    text = re.sub(r"\s+", " ",text)
    text = re.sub(r"<.*?>", " ",text)
    text = text.strip().lower()
    return text

In [11]:
train_data["dialogue"] = train_data["dialogue"].apply(clean_data)
train_data["summary"] = train_data["summary"].apply(clean_data)

val_data["dialogue"] = val_data["dialogue"].apply(clean_data)
val_data["summary"] = val_data["summary"].apply(clean_data)


In [12]:
train_data["dialogue"][0]

"violet: hi! i came across this austin's article and i thought that you might find it interesting violet:   claire: hi! :) thanks, but i've already read it. :) claire: but thanks for thinking about me :)"

# Tokenize

In [13]:
tokenizer = T5Tokenizer.from_pretrained("t5-small")

In [14]:
# raw data => tokenized inputs for fine- tuning

def tokeniz(data):
    inputs = tokenizer(data["dialogue"], padding = "max_length", max_length = 512, truncation = True)
    targets = tokenizer(data["summary"], padding = "max_length", max_length = 150, truncation = True)

    inputs["labels"] = targets["input_ids"]  # Token ids => ids input as label
    return inputs

In [15]:
train_dataset = train_data.apply(tokeniz, axis = 1).tolist()


In [16]:
val_dataset = val_data.apply(tokeniz, axis = 1).tolist()

In [17]:
train_dataset[0]

{'input_ids': [25208, 10, 7102, 55, 3, 23, 764, 640, 48, 403, 17, 77, 31, 7, 1108, 11, 3, 23, 816, 24, 25, 429, 253, 34, 1477, 25208, 10, 3, 7997, 15, 10, 7102, 55, 3, 10, 61, 2049, 6, 68, 3, 23, 31, 162, 641, 608, 34, 5, 3, 10, 61, 3, 7997, 15, 10, 68, 2049, 21, 1631, 81, 140, 3, 10, 61, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,

In [18]:
# input_ids - dialoge => token ids

# 1 => EOS, 0 => padding

# attention mask  - if 1 => token ids valid     if 0 => that is invalid that in training time
#labels = target ==> summary token

In [19]:
len(train_dataset[0]["input_ids"])
len(train_dataset[0]["labels"])


150

In [20]:
type(train_dataset)
type(val_data)

pandas.core.frame.DataFrame

# Working with the Model



In [21]:
# NLP ==> Generation task

model = T5ForConditionalGeneration.from_pretrained("t5-small")

Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]

In [22]:
print(torch.cuda.is_available())

True


In [23]:
!pip show torch

Name: torch
Version: 2.14.0+cu126
Summary: Tensors and Dynamic neural networks in Python with strong GPU acceleration
Home-page: https://pytorch.org
Author: 
Author-email: PyTorch Team <packages@pytorch.org>
License-Expression: Apache-2.0 AND Apache-2.0 WITH LLVM-exception AND BSD-2-Clause AND BSD-3-Clause AND BSL-1.0 AND MIT
Location: C:\Users\nites\anaconda3\Lib\site-packages
Requires: filelock, fsspec, jinja2, networkx, setuptools, sympy, typing-extensions
Required-by: accelerate, torchvision


In [24]:
# Fine -tune

import torch

if torch.backends.mps.is_available():
    device = torch.device("mps")
elif torch.cuda.is_available():
    device = torch.device("cuda")
else:
    device = torch.device("cpu")

print("device: " , device)
model.to(device)

device:  cuda


T5ForConditionalGeneration(
  (shared): Embedding(32128, 512)
  (encoder): T5Stack(
    (embed_tokens): Embedding(32128, 512)
    (block): ModuleList(
      (0): T5Block(
        (layer): ModuleList(
          (0): T5LayerSelfAttention(
            (SelfAttention): T5Attention(
              (q): Linear(in_features=512, out_features=512, bias=False)
              (k): Linear(in_features=512, out_features=512, bias=False)
              (v): Linear(in_features=512, out_features=512, bias=False)
              (o): Linear(in_features=512, out_features=512, bias=False)
              (relative_attention_bias): Embedding(32, 8)
            )
            (layer_norm): T5LayerNorm()
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (1): T5LayerFF(
            (DenseReluDense): T5DenseActDense(
              (wi): Linear(in_features=512, out_features=2048, bias=False)
              (wo): Linear(in_features=2048, out_features=512, bias=False)
              (dropout): Drop

In [25]:
# Training Arguments

training_args = TrainingArguments(
    output_dir = "./results",

    num_train_epochs = 6,
    weight_decay = 0.01,

    per_device_train_batch_size = 8,
    per_device_eval_batch_size = 8,

    eval_strategy = "epoch",
    save_strategy = "epoch",

    warmup_steps=500
    # 0  ==> lr default
)
    

In [26]:
trainer = Trainer(
    model = model,
    args = training_args,
    train_dataset = train_dataset,
    eval_dataset = val_dataset

)


In [27]:
# train the model

trainer.train()

Epoch,Training Loss,Validation Loss
1,3.645939,0.381597
2,0.396677,0.359936
3,0.373918,0.355015
4,0.363077,0.351001
5,0.356175,0.350157
6,0.351797,0.349737


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=3000, training_loss=0.9145973205566407, metrics={'train_runtime': 2380.0712, 'train_samples_per_second': 10.084, 'train_steps_per_second': 1.26, 'total_flos': 3248203235328000.0, 'train_loss': 0.9145973205566407, 'epoch': 6.0})

In [28]:
model.save_pretrained("./saved_summary_model")
tokenizer.save_pretrained("./saved_summary_model")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('./saved_summary_model\\tokenizer_config.json',
 './saved_summary_model\\tokenizer.json')

In [29]:
model = T5ForConditionalGeneration.from_pretrained("./saved_summary_model")
tokenizer = T5Tokenizer.from_pretrained("./saved_summary_model")

Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]

In [36]:
# def summarize_dialogue(dialogue):
#      #clean
#     dialogue = clean_data(dialogue)

#     # tokenize
#     inputs = tokenizer(
#         dialogue,
#         padding = "max_length",
#         max_length = 512,
#         truncation = True,
#         return_tensors = "pt"
#     )

#     #generate the summary => token ids
#     model.to(device)
#     targets = model.generate(
#         input_ids = inputs["input_ids"],
#         attention_mask = inputs["attention_mask"],
#         max_length = 150,
#         num_beams = 4,
#         early_stopping = True
#     )

#     # token ids convert to summary / decode our output
#     summary = tokenizer.decode(targets[0], skip_special_tokens = True)  #EOS SEP
#     return summary

def summarize_dialogue(dialoge):
    dialogue = clean_data(dialoge)  # Clean the input dialogue

    # Tokenize
    inputs = tokenizer(
        dialogue,
        padding="max_length",
        max_length=512,
        truncation=True,
        return_tensors="pt"
    )

    # Move inputs to the same device as the model
    inputs = {key: value.to(device) for key, value in inputs.items()}

    # Generate the summary
    model.to(device)

    targets = model.generate(
        input_ids=inputs["input_ids"],
        attention_mask=inputs["attention_mask"],
        max_length=150,
        num_beams=4,
        early_stopping=True
    )

    # Convert token IDs to text
    summary = tokenizer.decode(
        targets[0],
        skip_special_tokens=True
    )

    return summary

In [38]:
test_dialogue = """
Manager: Good morning, everyone. We need to discuss the progress of our mobile application.
Developer: The login and registration modules are completed, but the payment integration is still in progress.
Manager: What is causing the delay in the payment integration?
Developer: The payment gateway API has some compatibility issues with our backend. We are working with the provider to resolve them.
Designer: The new dashboard design is also ready. I will send the final version to the development team today.
Tester: I have tested the login module and found three minor bugs. They are related to password validation and error messages.
Developer: I can fix those bugs by tomorrow afternoon.
Manager: Good. We need to complete testing by Friday so that we can release the application next week.
Tester: If the payment integration is completed on time, I should be able to finish the remaining testing by Friday.
Manager: Excellent. Let's meet again on Thursday to review the final progress.
"""
summary = summarize_dialogue(test_dialogue)
print("Summary:", summary )

Summary: developer has tested the login module and found three minor bugs related to password validation and error messages. manager will send the final version to the development team today.
